In [1]:
import torch
import torch.optim as optim
import torch.nn as nn 
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW

import torchvision
import torchvision.transforms as transforms 
from torchvision import datasets

from pathlib import Path
from PIL import Image
import json
import random

In [2]:

# ! dirs
PROJECT_DIR = Path.cwd().parent
CHECKPOINTS_DIR = PROJECT_DIR / "checkpoints"
DATA_DIR = PROJECT_DIR / "data"
REPORTS_DIR = PROJECT_DIR / "reports"
SRC_DIR = PROJECT_DIR / "src"

# ! files
TRAIN_DATA = DATA_DIR / "train"
VAL_DATA = DATA_DIR / "val"
TEST_DATA = DATA_DIR / "test" / "images"

In [3]:
class TinyImageNetValDataset(Dataset):
    def __init__(self, val_dir, transform=None, class_to_idx=None):
        self.val_dir = Path(val_dir)
        self.images_dir = self.val_dir / "images"
        self.annotations_path = self.val_dir / "val_annotations.txt"
        self.transform = transform
        self.class_to_idx = class_to_idx

        self.samples = []

        with open(self.annotations_path, "r", encoding="utf-8") as file:
            for line in file:
                parts = line.strip().split()

                image_name = parts[0]
                wnid = parts[1]

                label = self.class_to_idx[wnid]

                image_path = self.images_dir / image_name

                self.samples.append((image_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomCrop(64, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.RandomErasing(p=0.25)
])
val_and_test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

In [ ]:
train_dataset = datasets.ImageFolder(
    root = TRAIN_DATA,
    transform=train_transform
)
val_dataset = TinyImageNetValDataset(
    val_dir=VAL_DATA,
    transform=val_and_test_transform,
    class_to_idx=train_dataset.class_to_idx
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


In [6]:
images, labels = next(iter(train_dataloader))

print(images.shape)
print(labels.shape)
print(images.dtype)
print(labels.dtype)
print(images.min().item())
print(labels.min().item())
print(labels[0:10])

images, labels = next(iter(val_dataloader))
print("\n",images.shape)
print(labels.shape)
print(images.dtype)
print(labels.dtype)
print(images.min().item())
print(labels.min().item())
print(labels[0:10])

torch.Size([256, 3, 64, 64])
torch.Size([256])
torch.float32
torch.int64
-1.0
0
tensor([180,  26, 131, 171,  29,  34, 121, 147, 166,   0])

 torch.Size([256, 3, 64, 64])
torch.Size([256])
torch.float32
torch.int64
-1.0
0
tensor([107, 139, 140,  69,  69, 161, 147,  73, 145,  39])


In [7]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 3 * 64 * 64

        # block 1
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)       # 3 * 64 * 64 -> 32 * 64 * 64
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 32, 3, padding=1)      # 32 * 64 * 64 -> 32 * 64 * 64
        self.bn2 = nn.BatchNorm2d(32)

        self.pool1 = nn.MaxPool2d(2, 2)                   # 32 * 64 * 64 -> 32 * 32 * 32


        # block 2
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)      # 32 * 32 * 32 -> 64 * 32 * 32
        self.bn3 = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)      # 64 * 32 * 32 -> 64 * 32 * 32
        self.bn4 = nn.BatchNorm2d(64)

        self.pool2 = nn.MaxPool2d(2, 2)                   # 64 * 32 * 32 -> 64 * 16 * 16


        # block 3
        self.conv5 = nn.Conv2d(64, 128, 3, padding=1)     # 64 * 16 * 16 -> 128 * 16 * 16
        self.bn5 = nn.BatchNorm2d(128)

        self.conv6 = nn.Conv2d(128, 128, 3, padding=1)    # 128 * 16 * 16 -> 128 * 16 * 16
        self.bn6 = nn.BatchNorm2d(128)

        self.pool3 = nn.MaxPool2d(2, 2)                   # 128 * 16 * 16 -> 128 * 8 * 8


        # block 4
        self.conv7 = nn.Conv2d(128, 256, 3, padding=1)    # 128 * 8 * 8 -> 256 * 8 * 8
        self.bn7 = nn.BatchNorm2d(256)

        self.conv8 = nn.Conv2d(256, 256, 3, padding=1)    # 256 * 8 * 8 -> 256 * 8 * 8
        self.bn8 = nn.BatchNorm2d(256)

        self.pool4 = nn.MaxPool2d(2, 2)                   # 256 * 8 * 8 -> 256 * 4 * 4


        # classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))       # 256 * 4 * 4 -> 256 * 1 * 1
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(256, 200)

    def forward(self, X):
        # block 1
        X = F.relu(self.bn1(self.conv1(X)))
        X = F.relu(self.bn2(self.conv2(X)))
        X = self.pool1(X)

        # block 2
        X = F.relu(self.bn3(self.conv3(X)))
        X = F.relu(self.bn4(self.conv4(X)))
        X = self.pool2(X)

        # block 3
        X = F.relu(self.bn5(self.conv5(X)))
        X = F.relu(self.bn6(self.conv6(X)))
        X = self.pool3(X)

        # block 4
        X = F.relu(self.bn7(self.conv7(X)))
        X = F.relu(self.bn8(self.conv8(X)))
        X = self.pool4(X)

        # classifier
        X = self.avgpool(X)
        X = torch.flatten(X, 1)
        X = self.dropout(X)
        X = self.fc1(X)

        return X

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN().to(device)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=100
)

In [ ]:
best_val_acc = 0.0
best_val_loss = float("inf")
best_epoch = 0

history = []

for epoch in range(100):
    print(f"Training epoch {epoch + 1}....")
    model.train()

    current_loss = 0.0
    correct = 0
    total = 0

    for num_batch, batch in enumerate(train_dataloader):
        images, labels = batch

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(images)
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        current_loss += loss.item()

        predicted = torch.argmax(logits, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = current_loss / len(train_dataloader)
    train_acc = 100 * correct / total

    model.eval()

    val_current_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for num_batch, batch in enumerate(val_dataloader):
            images, labels = batch

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = loss_fn(logits, labels)

            val_current_loss += loss.item()

            predicted = torch.argmax(logits, dim=1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = val_current_loss / len(val_dataloader)
    val_acc = 100 * val_correct / val_total

    epoch_result = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    }

    history.append(epoch_result)

    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(model.state_dict(), CHECKPOINTS_DIR / "best_model.pth")

        print("Best model saved")

    print(f"Train loss: {train_loss:.4f}")
    print(f"Train accuracy: {train_acc:.2f}%")
    print(f"Val loss: {val_loss:.4f}")
    print(f"Val accuracy: {val_acc:.2f}%")
    print("-" * 40)

Training epoch 1....


In [ ]:
metrics = {
    "best_epoch": best_epoch,
    "best_val_acc": best_val_acc,
    "best_val_loss": best_val_loss
}

with open(REPORTS_DIR / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=4)

with open(REPORTS_DIR / "history.json", "w", encoding="utf-8") as file:
    json.dump(history, file, indent=4)

In [ ]:
best_model = CNN()
best_model.load_state_dict(torch.load(CHECKPOINTS_DIR / "best_model.pth"))

<All keys matched successfully>

In [ ]:
best_model = CNN().to(device)
best_model.load_state_dict(
    torch.load(CHECKPOINTS_DIR / "best_model.pth", map_location=device)
)
best_model.eval()

def load_test_file(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_and_test_transform(image)
    image = image.unsqueeze(0)
    image = image.to(device)
    return image

wnid_to_name = {}

with open(DATA_DIR / "words.txt", "r", encoding="utf-8") as f:
    for line in f:
        wnid, class_name = line.strip().split("\t", 1)
        wnid_to_name[wnid] = class_name

idx_to_wnid = train_dataset.classes

image_paths = list(TEST_DATA.glob("*.JPEG"))
image_paths = random.sample(image_paths, 5)

with torch.no_grad():
    for image_path in image_paths:
        image = load_test_file(image_path)

        logits = best_model(image)
        predicted = torch.argmax(logits, dim=1)

        predicted_idx = predicted.item()
        predicted_wnid = idx_to_wnid[predicted_idx]
        predicted_name = wnid_to_name[predicted_wnid]

        print(f"Image: {image_path.name}")
        print(f"Predicted wnid: {predicted_wnid}")
        print(f"Prediction: {predicted_name}")
        print("-" * 40)

Image: test_1909.JPEG
Predicted wnid: n02963159
Prediction: cardigan
----------------------------------------
Image: test_9234.JPEG
Predicted wnid: n02125311
Prediction: cougar, puma, catamount, mountain lion, painter, panther, Felis concolor
----------------------------------------
Image: test_9626.JPEG
Predicted wnid: n02099601
Prediction: golden retriever
----------------------------------------
Image: test_8135.JPEG
Predicted wnid: n02125311
Prediction: cougar, puma, catamount, mountain lion, painter, panther, Felis concolor
----------------------------------------
Image: test_9413.JPEG
Predicted wnid: n03617480
Prediction: kimono
----------------------------------------
